# 有休残数推移 & 未消化リスク分析
## SQ・RASIEL施設 ― Google Drive連携版 v2

**使い方**
1. セルを **Shift+Enter** で実行（1つにまとまっています）
2. Googleアカウント認証を済ませてください
3. 新しい月のファイルをDriveの「有休残一覧」フォルダに追加するだけで自動反映

---
**出力物**
- 📊 月次推移テーブル（色付き）
- 🔴 未消化リスク判定（変化量・年換算ペース）
- 🟡 残数少アラート（5日以下）
- 📥 Excel出力（4シート構成：凡例・職員別推移・要注意職員・施設別平均）

In [ ]:
# ============================================================
# 有休残数推移 & 未消化リスク分析　SQ・RASIEL施設
# Google Drive連携版 v2  ―  全処理を1セルに統合
# ============================================================

# ── ライブラリインストール ───────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '--quiet', 'gdown', 'openpyxl', 'pandas'], check=True)

# ── Google Drive 認証 ────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()
print('✓ 認証完了')

# ============================================================
# 設定（変更が必要な場合はここだけ編集）
# ============================================================
FOLDER_ID       = '17fPa-70adHhd44qYOGyGW4i3pMPPS5UP'  # 「有休残一覧」フォルダID
TARGET_KEYWORDS = ['SQ', 'SJ', 'ラシエル', 'RASIEL']   # 抽出対象施設キーワード
ALERT_LOW       = 5    # 残数アラート閾値（日）
ALERT_PACE      = 5.0  # 年換算消化ペースの警告閾値（日/年）
OUTPUT_FILE     = '有休残推移_SQ_RASIEL.xlsx'

print('設定完了')
print(f'  フォルダID       : {FOLDER_ID}')
print(f'  対象キーワード   : {TARGET_KEYWORDS}')
print(f'  残数アラート閾値 : {ALERT_LOW}日以下')
print(f'  消化ペース警告   : 年換算{ALERT_PACE}日未満')

# ============================================================
# DriveフォルダのXLSXを自動一覧取得・ダウンロード
# ============================================================
import io, os, re
import pandas as pd
import openpyxl
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

creds, _ = google.auth.default()
drive_service = build('drive', 'v3', credentials=creds)

def list_xlsx_in_folder(folder_id):
    query = (
        f"'{folder_id}' in parents "
        "and mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet' "
        "and trashed=false "
        "and not name contains '有休残推移_'"  # 出力レポートを入力から除外
    )
    result = drive_service.files().list(
        q=query,
        fields='files(id, name, modifiedTime)',
        orderBy='modifiedTime'
    ).execute()
    return result.get('files', [])

def download_xlsx(file_id):
    request = drive_service.files().get_media(fileId=file_id)
    buf = io.BytesIO()
    downloader = MediaIoBaseDownload(buf, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buf.seek(0)
    return buf

files = list_xlsx_in_folder(FOLDER_ID)
print(f'フォルダ内XLSXファイル: {len(files)}件')
for f in files:
    print(f"  {f['name']}  (更新: {f['modifiedTime'][:10]})")

# ============================================================
# 月ラベル自動生成・全ファイル読み込み
# ============================================================
def extract_month_label(filename):
    m = re.search(r'(\d{4})年0?(\d{1,2})月', filename)
    if m:
        return f"{m.group(1)}年{int(m.group(2))}月"
    return filename.replace('.xlsx', '')

def load_paid_leave(xlsx_bytes, label):
    wb = openpyxl.load_workbook(xlsx_bytes, read_only=True)
    ws = wb.active
    rows = []
    header_found = False
    col_idx = {}
    for row in ws.iter_rows(values_only=True):
        if row is None:
            continue
        cells = [str(c).strip() if c is not None else '' for c in row]
        if not header_found:
            if '施設' in cells and '氏名' in cells:
                col_idx['施設'] = cells.index('施設')
                col_idx['氏名'] = cells.index('氏名')
                for cand in ['残数', '有休残', '残']:
                    if cand in cells:
                        col_idx['残数'] = cells.index(cand)
                        break
                if '残数' in col_idx:
                    header_found = True
            continue
        try:
            shisetsu = cells[col_idx['施設']]
            shimei   = cells[col_idx['氏名']]
            zansu    = row[col_idx['残数']]
        except IndexError:
            continue
        if not shisetsu or not shimei:
            continue
        try:
            zansu = float(zansu) if zansu is not None else None
        except (ValueError, TypeError):
            zansu = None
        rows.append({'施設': shisetsu, '氏名': shimei, label: zansu})
    wb.close()
    return pd.DataFrame(rows)

monthly_dfs  = []
month_labels = []

for f in files:
    label = extract_month_label(f['name'])
    if not re.match(r'\d{4}年\d{1,2}月$', label):
        print(f"⚠ 年月を判別できないためスキップ: {f['name']}")
        continue
    buf   = download_xlsx(f['id'])
    df    = load_paid_leave(buf, label)
    monthly_dfs.append(df)
    month_labels.append(label)
    print(f"✓ {label}  ({f['name']})  {len(df)}件")

print('\n読み込み完了')

# ============================================================
# SQ・RASIEL施設のみ抽出・月次結合
# ============================================================
def filter_target(df, keywords):
    pattern = '|'.join(re.escape(k) for k in keywords)
    return df[df['施設'].str.contains(pattern, na=False)].copy()

merged = None
for df in monthly_dfs:
    if df.empty:
        continue
    label_cols_candidates = [c for c in df.columns if c not in ('施設', '氏名')]
    if not label_cols_candidates:
        continue
    filtered = filter_target(df, TARGET_KEYWORDS)
    filtered = filtered.drop_duplicates(subset=['施設', '氏名'])
    if merged is None:
        merged = filtered
    else:
        merged = pd.merge(merged, filtered, on=['施設', '氏名'], how='outer')

if merged is None:
    merged = pd.DataFrame(columns=['施設', '氏名'])

merged = merged.sort_values(['施設', '氏名']).reset_index(drop=True)

actual_month_cols = [col for col in merged.columns if re.match(r'\d{4}年\d{1,2}月', col)]
month_labels = sorted(
    actual_month_cols,
    key=lambda x: (int(x.split('年')[0]), int(x.split('年')[1].replace('月', '')))
)

print(f'対象職員数: {len(merged)} 名')
print(f'施設一覧 : {merged["施設"].unique().tolist()}')

# ============================================================
# 付与月の自動検出・付与日数（推定）・付与後消化状況・リスク判定
#
# ■ 付与日数（推定）
#   前月比で残数が最も増えた月の増加幅を付与日数として自動検出。
#   付与と同じ月に消化があると「付与日数 − 当月消化」になるため「推定」と明示。
#   年10日以上付与（増加幅10日以上）の職員のみ5日取得義務の対象として判定。
# ============================================================
import numpy as np

last_col = month_labels[-1] if month_labels else None

def detect_grant_month(row):
    grant_month  = None
    max_increase = 0
    for i in range(1, len(month_labels)):
        prev_col = month_labels[i - 1]
        curr_col = month_labels[i]
        v_prev   = row.get(prev_col)
        v_curr   = row.get(curr_col)
        if v_prev is None or v_curr is None:
            continue
        if pd.isna(v_prev) or pd.isna(v_curr):
            continue
        increase = v_curr - v_prev
        if increase > max_increase:
            max_increase = increase
            grant_month  = curr_col
    return grant_month, max_increase

grant_results            = merged.apply(detect_grant_month, axis=1)
merged['付与月']         = grant_results.apply(lambda x: x[0] if x[0] else '―')
merged['付与時増加幅']   = grant_results.apply(lambda x: x[1])

# 付与日数（推定）列：増加幅が1日以上の場合のみ表示
merged['付与日数（推定）'] = merged['付与時増加幅'].apply(
    lambda x: round(x, 1) if (pd.notna(x) and x > 0) else None
)

def calc_grant_stats(row):
    grant_month    = row.get('付与月')
    grant_increase = row.get('付与時増加幅', 0)
    if (grant_month == '―' or grant_month is None
            or pd.isna(grant_increase) or grant_increase < 10):
        return pd.Series({'付与後消化日数': None, '付与後経過月数': None, '5日取得見込み': '―'})
    v_grant = row.get(grant_month)
    v_last  = row.get(last_col)
    if v_grant is None or v_last is None or pd.isna(v_grant) or pd.isna(v_last):
        return pd.Series({'付与後消化日数': None, '付与後経過月数': None, '5日取得見込み': '―'})
    consumed  = round(v_grant - v_last, 1)
    grant_idx = month_labels.index(grant_month)
    last_idx  = len(month_labels) - 1
    elapsed   = last_idx - grant_idx
    if consumed >= 5:
        outlook = '◎ 達成済み'
    elif elapsed == 0:
        outlook = '― 経過待ち'
    elif consumed <= 0:
        outlook = '✗ 未消化'
    else:
        annual_pace = consumed / elapsed * 12
        outlook = '○ 達成見込あり' if annual_pace >= 5 else '△ 達成困難'
    return pd.Series({'付与後消化日数': consumed, '付与後経過月数': elapsed, '5日取得見込み': outlook})

grant_stats = merged.apply(calc_grant_stats, axis=1)
merged      = pd.concat([merged, grant_stats], axis=1)

def calc_risk(row):
    vals = [(col, row[col]) for col in month_labels
            if col in row and row[col] is not None and not pd.isna(row[col])]
    if len(vals) < 2:
        v_last    = vals[0][1] if vals else None
        low_alert = v_last is not None and v_last <= ALERT_LOW
        return pd.Series({'変化量(消化+)': None, '年換算消化ペース': None,
                          '判定': '🟡 残数少' if low_alert else '―'})
    v_first         = vals[0][1]
    v_last          = vals[-1][1]
    elapsed_months = len(vals) - 1  # 経過月数（データ点数−1）
    change      = round(v_first - v_last, 1)
    annual_pace = round(change / elapsed_months * 12, 1)
    low_alert   = v_last <= ALERT_LOW
    if change <= 0 and low_alert:
        risk = '🔴 未消化＋残少'
    elif change <= 0:
        risk = '🔴 未消化リスク'
    elif annual_pace < ALERT_PACE:
        risk = '🟡 消化ペース不足'
    elif low_alert:
        risk = '🟡 残数少'
    else:
        risk = '🟢 OK'
    return pd.Series({'変化量(消化+)': change, '年換算消化ペース': annual_pace, '判定': risk})

risk_cols = merged.apply(calc_risk, axis=1)
result    = pd.concat([merged, risk_cols], axis=1)

print('計算完了')
print(result[[
    '施設', '氏名', '付与月', '付与日数（推定）',
    '付与後消化日数', '付与後経過月数', '5日取得見込み', '判定'
]].to_string(index=False))

# ============================================================
# 色付き推移テーブルの表示（Pandas Styler）
# ============================================================
from IPython.display import display

display_cols = (
    ['施設', '氏名', '付与月', '付与日数（推定）'] +
    month_labels +
    ['付与後消化日数', '付与後経過月数', '5日取得見込み',
     '変化量(消化+)', '年換算消化ペース', '判定']
)

def color_zansu(val):
    if pd.isna(val) or not isinstance(val, (int, float)):
        return ''
    if val <= 0:
        return 'background-color: #FF9999; color: #8B0000; font-weight: bold'
    elif val <= ALERT_LOW:
        return 'background-color: #FFE066; color: #7B6200'
    elif val >= 30:
        return 'background-color: #D4EDDA; color: #155724'
    else:
        return 'background-color: #EAF4FF; color: #004085'

def color_risk(val):
    if '🔴' in str(val):
        return 'background-color: #FF9999; font-weight: bold'
    elif '🟡' in str(val):
        return 'background-color: #FFE066'
    elif '🟢' in str(val):
        return 'background-color: #D4EDDA'
    return ''

def color_outlook(val):
    s = str(val)
    if '◎' in s: return 'background-color: #D4EDDA; font-weight: bold'
    elif '○' in s: return 'background-color: #EAF4FF'
    elif '△' in s: return 'background-color: #FFE066'
    elif '✗' in s: return 'background-color: #FF9999; font-weight: bold'
    return ''

table_styles = [
    {'selector': 'caption',
     'props': [('font-size', '16px'), ('font-weight', 'bold'),
               ('color', '#333'), ('padding', '8px 0')]},
    {'selector': 'th',
     'props': [('background-color', '#4472C4'), ('color', 'white'),
               ('font-weight', 'bold'), ('text-align', 'center'),
               ('padding', '6px 10px')]},
    {'selector': 'td',
     'props': [('text-align', 'center'), ('padding', '5px 10px'),
               ('border', '1px solid #ddd')]},
    {'selector': 'tr:hover td',
     'props': [('filter', 'brightness(0.95)')]},
]

num_fmt_cols = month_labels + ['付与後消化日数', '付与後経過月数', '変化量(消化+)', '年換算消化ペース']

for facility in result['施設'].unique():
    df_f    = result[result['施設'] == facility].copy()
    df_disp = df_f[display_cols].reset_index(drop=True)
    styled = (
        df_disp.style
        .map(color_zansu,  subset=month_labels)
        .map(color_risk,   subset=['判定'])
        .map(color_outlook, subset=['5日取得見込み'])
        .set_caption(f'【{facility}】有休残数推移')
        .set_table_styles(table_styles)
        .format(
            na_rep='―',
            subset=num_fmt_cols,
            formatter=lambda v: f'{v:.1f}' if isinstance(v, float) else v
        )
    )
    display(styled)
    print()

# ============================================================
# リスク一覧サマリー（要注意職員のみ）
# ============================================================
from IPython.display import HTML

warn_df = result[result['判定'].str.contains('🔴|🟡', na=False)].copy()
warn_df = warn_df[['施設', '氏名'] + month_labels + ['変化量(消化+)', '年換算消化ペース', '判定']]
warn_df = warn_df.sort_values(['判定', '施設', '氏名']).reset_index(drop=True)

if warn_df.empty:
    print('✓ 要注意職員なし')
else:
    print(f'⚠ 要注意職員: {len(warn_df)} 名')
    styled_warn = (
        warn_df.style
        .map(color_zansu, subset=month_labels)
        .map(color_risk, subset=['判定'])
        .set_caption('【要注意職員一覧】🔴 未消化リスク  🟡 消化ペース不足・残数少')
        .set_table_styles([{
            'selector': 'caption',
            'props': [('font-size', '16px'), ('font-weight', 'bold'),
                      ('color', '#C00000'), ('padding', '8px 0')]
        }, {
            'selector': 'th',
            'props': [('background-color', '#C00000'), ('color', 'white'),
                      ('font-weight', 'bold'), ('text-align', 'center'),
                      ('padding', '6px 10px')]
        }, {
            'selector': 'td',
            'props': [('text-align', 'center'), ('padding', '5px 10px'),
                      ('border', '1px solid #ddd')]
        }])
        .format(na_rep='―', subset=month_labels + ['変化量(消化+)', '年換算消化ペース'],
                formatter=lambda v: f'{v:.1f}' if isinstance(v, float) else v)
    )
    display(styled_warn)

# ============================================================
# Excel出力（色付き＋凡例・説明シート付き）
# ============================================================
from google.colab import files
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def get_fill(val):
    try:
        num_val = float(val)
    except (ValueError, TypeError):
        return None
    if pd.isna(num_val):
        return None
    if num_val <= 0:
        return PatternFill('solid', fgColor='FF9999')
    elif num_val <= ALERT_LOW:
        return PatternFill('solid', fgColor='FFE066')
    elif num_val >= 30:
        return PatternFill('solid', fgColor='D4EDDA')
    else:
        return PatternFill('solid', fgColor='EAF4FF')

def get_risk_fill(val):
    if '🔴' in str(val): return PatternFill('solid', fgColor='FF9999')
    elif '🟡' in str(val): return PatternFill('solid', fgColor='FFE066')
    elif '🟢' in str(val): return PatternFill('solid', fgColor='D4EDDA')
    return None

def get_outlook_fill(val):
    s = str(val)
    if '◎' in s: return PatternFill('solid', fgColor='D4EDDA')
    elif '○' in s: return PatternFill('solid', fgColor='EAF4FF')
    elif '△' in s: return PatternFill('solid', fgColor='FFE066')
    elif '✗' in s: return PatternFill('solid', fgColor='FF9999')
    return None

thin        = Side(style='thin', color='AAAAAA')
border      = Border(left=thin, right=thin, top=thin, bottom=thin)
header_fill = PatternFill('solid', fgColor='4472C4')
header_font = Font(bold=True, color='FFFFFF')
center      = Alignment(horizontal='center', vertical='center')
left_wrap   = Alignment(horizontal='left', vertical='center', wrap_text=True)

# 列インデックスの計算
# display_cols = ['施設','氏名','付与月','付与日数（推定）'] + month_labels + [...]
month_col_start = 5                          # E列から月データ（A〜D=施設・氏名・付与月・付与日数）
month_col_end   = 4 + len(month_labels)
outlook_col_idx = month_col_end + 3          # 5日取得見込み列
risk_col_idx    = len(display_cols)          # 判定列（最終列）

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:

    # ── シート1: 職員別推移 ──────────────────────────────
    result[display_cols].to_excel(writer, sheet_name='職員別推移', index=False)
    ws1 = writer.sheets['職員別推移']
    for cell in ws1[1]:
        cell.fill = header_fill; cell.font = header_font
        cell.alignment = center; cell.border = border
    for row in ws1.iter_rows(min_row=2):
        for ci, cell in enumerate(row, start=1):
            cell.border = border; cell.alignment = center
            if month_col_start <= ci <= month_col_end:
                f = get_fill(cell.value)
                if f: cell.fill = f
            elif ci == outlook_col_idx:
                f = get_outlook_fill(cell.value)
                if f: cell.fill = f
            elif ci == risk_col_idx:
                f = get_risk_fill(cell.value)
                if f: cell.fill = f
    ws1.column_dimensions['A'].width = 18
    ws1.column_dimensions['B'].width = 16
    ws1.column_dimensions['C'].width = 14
    ws1.column_dimensions['D'].width = 16  # 付与日数（推定）
    for i in range(len(month_labels)):
        ws1.column_dimensions[get_column_letter(5 + i)].width = 12
    offset = 5 + len(month_labels)
    ws1.column_dimensions[get_column_letter(offset)].width     = 16
    ws1.column_dimensions[get_column_letter(offset + 1)].width = 16
    ws1.column_dimensions[get_column_letter(offset + 2)].width = 18
    ws1.column_dimensions[get_column_letter(offset + 3)].width = 16
    ws1.column_dimensions[get_column_letter(offset + 4)].width = 18
    ws1.column_dimensions[get_column_letter(offset + 5)].width = 22

    # ── シート2: 要注意職員 ──────────────────────────────
    warn_df2 = result[result['判定'].str.contains('🔴|🟡', na=False)].copy()
    warn_df2 = warn_df2[display_cols].sort_values(['判定', '施設', '氏名']).reset_index(drop=True)
    if not warn_df2.empty:
        warn_df2.to_excel(writer, sheet_name='要注意職員', index=False)
        ws2 = writer.sheets['要注意職員']
        for cell in ws2[1]:
            cell.fill = PatternFill('solid', fgColor='C00000')
            cell.font = Font(bold=True, color='FFFFFF')
            cell.alignment = center; cell.border = border
        for row in ws2.iter_rows(min_row=2):
            for ci, cell in enumerate(row, start=1):
                cell.border = border; cell.alignment = center
                if month_col_start <= ci <= month_col_end:
                    f = get_fill(cell.value)
                    if f: cell.fill = f
                elif ci == outlook_col_idx:
                    f = get_outlook_fill(cell.value)
                    if f: cell.fill = f
                elif ci == risk_col_idx:
                    f = get_risk_fill(cell.value)
                    if f: cell.fill = f
        ws2.column_dimensions['A'].width = 18
        ws2.column_dimensions['B'].width = 16
        ws2.column_dimensions['C'].width = 14
        ws2.column_dimensions['D'].width = 16

    # ── シート3: 施設別平均 ──────────────────────────────
    summary = result.groupby('施設')[month_labels].mean().round(1)
    summary.to_excel(writer, sheet_name='施設別平均')
    ws3 = writer.sheets['施設別平均']
    for cell in ws3[1]:
        cell.fill = header_fill; cell.font = header_font
        cell.alignment = center; cell.border = border
    ws3.column_dimensions['A'].width = 20
    for i in range(len(month_labels)):
        ws3.column_dimensions[get_column_letter(2 + i)].width = 14

    # ── シート4: 凡例・説明 ──────────────────────────────
    wb        = writer.book
    ws_legend = wb.create_sheet(title='凡例・説明')

    title_font   = Font(bold=True, size=14, color='FFFFFF')
    title_fill   = PatternFill('solid', fgColor='1F3864')
    section_font = Font(bold=True, size=11, color='FFFFFF')
    section_fill = PatternFill('solid', fgColor='4472C4')
    body_font    = Font(size=10)
    label_font   = Font(bold=True, size=10)
    gray_fill    = PatternFill('solid', fgColor='F2F2F2')

    ws_legend.column_dimensions['A'].width = 4
    ws_legend.column_dimensions['B'].width = 22
    ws_legend.column_dimensions['C'].width = 48
    ws_legend.column_dimensions['D'].width = 4

    def wc(ws, row, col, value, font=None, fill=None, alignment=None,
           border=None, height=None):
        c = ws.cell(row, col, value)
        if font:      c.font      = font
        if fill:      c.fill      = fill
        if alignment: c.alignment = alignment
        if border:    c.border    = border
        if height:    ws.row_dimensions[row].height = height
        return c

    r = 1
    ws_legend.merge_cells(f'B{r}:C{r}')
    wc(ws_legend, r, 2, '有休残数推移レポート　凡例・見方ガイド',
       font=title_font, fill=title_fill,
       alignment=Alignment(horizontal='center', vertical='center'), height=30)
    r += 1
    ws_legend.row_dimensions[r].height = 8; r += 1

    ws_legend.merge_cells(f'B{r}:C{r}')
    wc(ws_legend, r, 2, '① シート構成',
       font=section_font, fill=section_fill,
       alignment=Alignment(horizontal='left', vertical='center', indent=1), height=22)
    r += 1
    for name, desc in [
        ('凡例・説明',  'このシート。見方・ロジックの説明'),
        ('職員別推移',  '全職員の月別有休残数と判定結果（色付き）'),
        ('要注意職員',  '🔴🟡 判定の職員のみ抽出したシート'),
        ('施設別平均',  '施設ごとの月別平均有休残数'),
    ]:
        wc(ws_legend, r, 2, f'【{name}】', font=label_font,
           alignment=Alignment(horizontal='left', vertical='center', indent=1), height=18)
        wc(ws_legend, r, 3, desc, font=body_font,
           alignment=Alignment(horizontal='left', vertical='center'))
        r += 1
    ws_legend.row_dimensions[r].height = 8; r += 1

    ws_legend.merge_cells(f'B{r}:C{r}')
    wc(ws_legend, r, 2, '② セルの色の意味（月別有休残数欄）',
       font=section_font, fill=section_fill,
       alignment=Alignment(horizontal='left', vertical='center', indent=1), height=22)
    r += 1
    for hex_color, label, desc in [
        ('FF9999', '0日（有休なし）',            '即対応が必要。有休がない状態。'),
        ('FFE066', f'1〜{ALERT_LOW}日（残少）',  f'残数が少ない。{ALERT_LOW}日以下はアラート対象。取得促進が必要。'),
        ('EAF4FF', '6〜29日（通常）',            '標準的な残数の範囲。'),
        ('D4EDDA', '30日以上（積み残し多）',     '残数が多すぎる可能性。消化促進の声かけを検討。'),
    ]:
        c = ws_legend.cell(r, 2, label)
        c.fill = PatternFill('solid', fgColor=hex_color)
        c.font = Font(bold=True, size=10)
        c.alignment = Alignment(horizontal='center', vertical='center')
        c.border = border
        wc(ws_legend, r, 3, desc, font=body_font, alignment=left_wrap, border=border)
        ws_legend.row_dimensions[r].height = 20; r += 1
    ws_legend.row_dimensions[r].height = 8; r += 1

    ws_legend.merge_cells(f'B{r}:C{r}')
    wc(ws_legend, r, 2, '③ 判定マークの意味',
       font=section_font, fill=section_fill,
       alignment=Alignment(horizontal='left', vertical='center', indent=1), height=22)
    r += 1
    for mark, hex_color, desc in [
        ('🔴 未消化＋残少',   'FF9999', '観測期間中に残数が減っておらず、かつ残数が5日以下。最優先で対応が必要。'),
        ('🔴 未消化リスク',   'FF9999', '観測期間中に残数がほぼ変わっていない、または増加している。有休を取得していない可能性が高い。'),
        ('🟡 消化ペース不足', 'FFE066', '消化はしているが、年換算で5日/年に満たないペース。労基法上の義務達成が困難な恐れあり。'),
        ('🟡 残数少',         'FFE066', '消化ペースは問題ないが、現時点の残数が5日以下。'),
        ('🟢 OK',             'D4EDDA', '消化ペース・残数ともに問題なし。'),
        ('―',                'F2F2F2', 'データが1ヶ月分のみで判定不能。'),
    ]:
        c = ws_legend.cell(r, 2, mark)
        c.fill = PatternFill('solid', fgColor=hex_color)
        c.font = Font(bold=True, size=10)
        c.alignment = Alignment(horizontal='center', vertical='center')
        c.border = border
        wc(ws_legend, r, 3, desc, font=body_font, alignment=left_wrap, border=border)
        ws_legend.row_dimensions[r].height = 30; r += 1
    ws_legend.row_dimensions[r].height = 8; r += 1

    ws_legend.merge_cells(f'B{r}:C{r}')
    wc(ws_legend, r, 2, '④ 付与日数（推定）・付与後消化・5日取得見込みの計算ロジック',
       font=section_font, fill=section_fill,
       alignment=Alignment(horizontal='left', vertical='center', indent=1), height=22)
    r += 1
    for label, desc in [
        ('付与月',
         '前月より残数が増えた月を有休付与発生月として自動検出。\n'
         'データは1年以内の運用のため、付与は1回のみ検出される前提。'),
        ('付与日数（推定）',
         '付与月の前月比増加幅を付与日数として表示。\n'
         '付与と同じ月に消化があると「付与日数 − 当月消化」の値になるため「推定」と明示。\n'
         '年10日以上付与の職員のみ5日取得義務の対象として判定。'),
        ('付与後消化日数',
         '付与月の残数 − 最新月の残数\n'
         '例）付与月20日 → 最新月17日 の場合： 20 - 17 ＝ 3.0日消化'),
        ('付与後経過月数',
         '付与月から最新月までの月数。\n'
         '例）4月付与 → 7月最新 の場合： 3ヶ月経過'),
        ('5日取得見込み（記号の意味）',
         '◎ 達成済み    ：既に5日以上消化済み\n'
         '○ 達成見込あり：現ペースで年換算5日以上になる見込み\n'
         '△ 達成困難    ：現ペースでは年5日に届かない恐れあり\n'
         '✗ 未消化      ：付与後まったく消化していない\n'
         '― 経過待ち    ：付与月が最新月のため判定不能'),
        ('注意点',
         '・観測期間内に付与が検出できない職員は「―」表示\n'
         '・付与前からの繰り越し残数と付与分の区別はデータ上できない'),
    ]:
        lc = ws_legend.cell(r, 2, label)
        lc.font      = Font(bold=True, size=10)
        lc.fill      = gray_fill
        lc.alignment = Alignment(horizontal='left', vertical='top', indent=1, wrap_text=True)
        lc.border    = border
        dc = ws_legend.cell(r, 3, desc)
        dc.font      = body_font
        dc.alignment = left_wrap
        dc.border    = border
        lines = desc.count('\n') + 1
        ws_legend.row_dimensions[r].height = max(20, lines * 16)
        r += 1
    ws_legend.row_dimensions[r].height = 8; r += 1

    ws_legend.merge_cells(f'B{r}:C{r}')
    wc(ws_legend, r, 2, '⑤ 判定別　対応の目安',
       font=section_font, fill=section_fill,
       alignment=Alignment(horizontal='left', vertical='center', indent=1), height=22)
    r += 1
    for mark, action in [
        ('🔴 未消化リスク',
         '面談等で有休取得の意向を確認し、必要に応じて取得日を会社側で指定する（時季指定義務）。'),
        ('🟡 消化ペース不足',
         '現在の取得状況を確認し、年度内に5日以上取得できるよう計画的に取得を促す。'),
        ('🟡 残数少',
         '残数が少ないことを本人に伝え、急な事情に備えた残数管理を意識させる。'),
        ('🟢 OK',
         '引き続き状況をモニタリング。30日以上の場合は積み残しが多いため消化促進を検討。'),
        ('△ 達成困難',
         '付与後の消化ペースが低い。年5日取得義務の達成に向けて早めに取得を促す。'),
        ('✗ 未消化',
         '付与後まったく消化していない。速やかに取得計画を立てるよう指導する。'),
    ]:
        mc = ws_legend.cell(r, 2, mark)
        mc.font      = Font(bold=True, size=10)
        mc.alignment = Alignment(horizontal='left', vertical='top', indent=1, wrap_text=True)
        mc.border    = border
        ac = ws_legend.cell(r, 3, action)
        ac.font      = body_font
        ac.alignment = left_wrap
        ac.border    = border
        lines = action.count('\n') + 1
        ws_legend.row_dimensions[r].height = max(22, lines * 18)
        r += 1

    ws_legend.row_dimensions[r].height = 8; r += 1
    ws_legend.merge_cells(f'B{r}:C{r}')
    fc = ws_legend.cell(r, 2,
        f'※ アラート閾値：残数{ALERT_LOW}日以下 ／ 消化ペース年{ALERT_PACE}日未満　'
        f'（設定変更はノートブック冒頭の設定セクションで行う）')
    fc.font      = Font(size=9, color='888888', italic=True)
    fc.alignment = Alignment(horizontal='left', vertical='center')
    ws_legend.row_dimensions[r].height = 16

# withブロックを抜けた後にシート順を変更（凡例・説明を先頭へ）
wb2 = openpyxl.load_workbook(OUTPUT_FILE)
if '凡例・説明' in wb2.sheetnames:
    idx = wb2.sheetnames.index('凡例・説明')
    wb2.move_sheet('凡例・説明', offset=-idx)
wb2.save(OUTPUT_FILE)

print(f"✓ '{OUTPUT_FILE}' を出力しました")
print(f"  シート: 凡例・説明 / 職員別推移 / 要注意職員 / 施設別平均")
files.download(OUTPUT_FILE)

# ============================================================
# （任意）出力ExcelをDriveの「有休残一覧」フォルダに保存
# 不要な場合は以下をコメントアウトしてください
# ============================================================
from googleapiclient.http import MediaFileUpload

# 最新月をファイル名に付けて履歴を残す（例: 有休残推移_SQ_RASIEL_2026年7月.xlsx）
upload_name = f"有休残推移_SQ_RASIEL_{last_col}.xlsx" if last_col else OUTPUT_FILE

media = MediaFileUpload(
    OUTPUT_FILE,
    mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
)

# 同名ファイルが既にあれば上書き（増殖防止）、なければ新規作成
existing = drive_service.files().list(
    q=f"'{FOLDER_ID}' in parents and name='{upload_name}' and trashed=false",
    fields='files(id)'
).execute().get('files', [])

if existing:
    uploaded = drive_service.files().update(
        fileId=existing[0]['id'], media_body=media, fields='id, name'
    ).execute()
    print(f"✓ Driveの既存ファイルを上書きしました: {upload_name} (ID: {uploaded['id']})")
else:
    uploaded = drive_service.files().create(
        body={'name': upload_name, 'parents': [FOLDER_ID]},
        media_body=media,
        fields='id, name'
    ).execute()
    print(f"✓ Driveに保存しました: {upload_name} (ID: {uploaded['id']})")
